In [1]:
import pandas as pd
from pathlib import Path

# Path to processed data
processed_dir = Path("../data/processed")

# Load datasets
train_df = pd.read_csv(processed_dir / "train.csv")
val_df = pd.read_csv(processed_dir / "validation.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

Train shape: (800, 12)
Validation shape: (100, 12)
Test shape: (100, 12)

Columns:
['pmid', 'question', 'context', 'long_answer', 'final_decision', 'year', 'question_words', 'context_words', 'answer_words', 'year_numeric', 'source', 'target']


In [2]:
from pathlib import Path

# Create a text file containing only training data
bpe_corpus_path = processed_dir / "bpe_training_corpus.txt"

with open(bpe_corpus_path, "w", encoding="utf-8") as f:
    for text in train_df["source"]:
        f.write(str(text) + "\n")

    for text in train_df["target"]:
        f.write(str(text) + "\n")

print("BPE training corpus created.")
print("Path:", bpe_corpus_path)
print("File exists:", bpe_corpus_path.exists())

BPE training corpus created.
Path: ..\data\processed\bpe_training_corpus.txt
File exists: True


In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Create an empty BPE tokenizer
bpe_tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

# Split text into basic pieces before BPE learns subwords
bpe_tokenizer.pre_tokenizer = Whitespace()

# Special tokens required for our Transformer
special_tokens = [
    "<PAD>",
    "<UNK>",
    "<BOS>",
    "<EOS>",
    "<SEP>"
]

# BPE trainer
trainer = BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=special_tokens
)

# Train BPE on training data only
bpe_tokenizer.train(
    [str(bpe_corpus_path)],
    trainer
)

print("BPE tokenizer trained successfully.")
print("BPE vocabulary size:", bpe_tokenizer.get_vocab_size())

BPE tokenizer trained successfully.
BPE vocabulary size: 8000


In [4]:
test_words = [
    "NCT01046032",
    "perimedullary",
    "Ketamine",
    "Atropine",
    "Aquagenic",
    "COVID-19",
    "IL-6",
    "α-synuclein"
]

print("BPE tokenization examples")
print("-" * 50)

for word in test_words:
    encoding = bpe_tokenizer.encode(word)
    print(f"{word:20} → {encoding.tokens}")

BPE tokenization examples
--------------------------------------------------
NCT01046032          → ['N', 'CT', '010', '4', '60', '32']
perimedullary        → ['perim', 'edullary']
Ketamine             → ['K', 'et', 'amine']
Atropine             → ['A', 'trop', 'ine']
Aquagenic            → ['A', 'qu', 'agen', 'ic']
COVID-19             → ['CO', 'V', 'ID', '-', '19']
IL-6                 → ['IL', '-', '6']
α-synuclein          → ['α', '-', 'syn', 'uc', 'le', 'in']


In [5]:
sample_source = train_df.iloc[0]["source"]
sample_target = train_df.iloc[0]["target"]

source_encoding = bpe_tokenizer.encode(sample_source)
target_encoding = bpe_tokenizer.encode(sample_target)

print("SOURCE")
print("-" * 50)
print(sample_source[:500])

print("\nBPE SOURCE TOKENS")
print("-" * 50)
print(source_encoding.tokens[:80])

print("\nTARGET")
print("-" * 50)
print(sample_target)

print("\nBPE TARGET TOKENS")
print("-" * 50)
print(target_encoding.tokens[:80])

print("\nToken counts:")
print("Source:", len(source_encoding.tokens))
print("Target:", len(target_encoding.tokens))

SOURCE
--------------------------------------------------
question: Visceral adipose tissue area measurement at a single level: can it represent visceral adipose tissue volume? context: Measurement of visceral adipose tissue (VAT) needs to be accurate and sensitive to change for risk monitoring. The purpose of this study is to determine the CT slice location where VAT area can best reflect changes in VAT volume and body weight. 60 plain abdominal CT images from 30 males [mean age (range) 51 (41-68) years, mean body weight (range) 71.1 (101.9-50.9) kg] 

BPE SOURCE TOKENS
--------------------------------------------------
['question', ':', 'V', 'is', 'c', 'eral', 'adipose', 'tissue', 'area', 'measurement', 'at', 'a', 'single', 'level', ':', 'can', 'it', 'represent', 'vis', 'c', 'eral', 'adipose', 'tissue', 'volume', '?', 'context', ':', 'Measure', 'ment', 'of', 'vis', 'c', 'eral', 'adipose', 'tissue', '(', 'VAT', ')', 'needs', 'to', 'be', 'accurate', 'and', 'sensitive', 'to', 'change', 

In [6]:
import numpy as np

source_lengths = []
target_lengths = []

for text in train_df["source"]:
    source_lengths.append(len(bpe_tokenizer.encode(text).ids))

for text in train_df["target"]:
    target_lengths.append(len(bpe_tokenizer.encode(text).ids))

print("BPE sequence length statistics")
print("-" * 50)

print("\nSOURCE")
print("Minimum:", min(source_lengths))
print("Maximum:", max(source_lengths))
print("Mean:", np.mean(source_lengths))
print("Median:", np.median(source_lengths))
print("90th percentile:", np.percentile(source_lengths, 90))
print("95th percentile:", np.percentile(source_lengths, 95))
print("99th percentile:", np.percentile(source_lengths, 99))

print("\nTARGET")
print("Minimum:", min(target_lengths))
print("Maximum:", max(target_lengths))
print("Mean:", np.mean(target_lengths))
print("Median:", np.median(target_lengths))
print("90th percentile:", np.percentile(target_lengths, 90))
print("95th percentile:", np.percentile(target_lengths, 95))
print("99th percentile:", np.percentile(target_lengths, 99))

BPE sequence length statistics
--------------------------------------------------

SOURCE
Minimum: 93
Maximum: 640
Mean: 313.01125
Median: 308.0
90th percentile: 413.20000000000005
95th percentile: 452.0999999999999
99th percentile: 560.06

TARGET
Minimum: 12
Maximum: 142
Mean: 49.965
Median: 47.0
90th percentile: 79.0
95th percentile: 90.04999999999995
99th percentile: 121.00999999999999


In [7]:
# Maximum sequence lengths for the BPE-based dataset

SOURCE_MAX_LENGTH_BPE = 512
TARGET_MAX_LENGTH_BPE = 128

print("BPE source maximum length:", SOURCE_MAX_LENGTH_BPE)
print("BPE target maximum length:", TARGET_MAX_LENGTH_BPE)

# Check how many training samples exceed these limits
source_truncated = sum(
    length > SOURCE_MAX_LENGTH_BPE
    for length in source_lengths
)

target_truncated = sum(
    length > TARGET_MAX_LENGTH_BPE
    for length in target_lengths
)

print("\nPotential truncation:")
print(
    f"Source: {source_truncated}/{len(source_lengths)} "
    f"({source_truncated / len(source_lengths) * 100:.2f}%)"
)

print(
    f"Target: {target_truncated}/{len(target_lengths)} "
    f"({target_truncated / len(target_lengths) * 100:.2f}%)"
)

BPE source maximum length: 512
BPE target maximum length: 128

Potential truncation:
Source: 17/800 (2.12%)
Target: 7/800 (0.88%)


In [8]:
def encode_bpe_text(text, max_length):
    """
    Convert text into BPE token IDs.

    Steps:
    1. Tokenize text using the trained BPE tokenizer.
    2. Convert tokens to their integer IDs.
    3. Truncate to the maximum length.
    4. Return the token IDs.
    """
    encoding = bpe_tokenizer.encode(str(text))

    token_ids = encoding.ids[:max_length]

    return token_ids


# Test on one training example
sample_ids = encode_bpe_text(
    train_df.iloc[0]["source"],
    SOURCE_MAX_LENGTH_BPE
)

print("Number of BPE IDs:", len(sample_ids))
print("First 30 IDs:", sample_ids[:30])

Number of BPE IDs: 217
First 30 IDs: [252, 28, 55, 137, 64, 712, 6683, 1599, 1560, 2055, 131, 62, 1427, 412, 28, 563, 148, 3483, 583, 64, 712, 6683, 1599, 1071, 33, 272, 28, 6328, 199, 134]


In [9]:
# Get special token IDs directly from the BPE tokenizer

BPE_PAD_ID = bpe_tokenizer.token_to_id("<PAD>")
BPE_UNK_ID = bpe_tokenizer.token_to_id("<UNK>")
BPE_BOS_ID = bpe_tokenizer.token_to_id("<BOS>")
BPE_EOS_ID = bpe_tokenizer.token_to_id("<EOS>")
BPE_SEP_ID = bpe_tokenizer.token_to_id("<SEP>")

print("BPE special token IDs")
print("-" * 30)
print("PAD:", BPE_PAD_ID)
print("UNK:", BPE_UNK_ID)
print("BOS:", BPE_BOS_ID)
print("EOS:", BPE_EOS_ID)
print("SEP:", BPE_SEP_ID)


def pad_sequence(token_ids, max_length, pad_id):
    """
    Pad or truncate a sequence to a fixed length.
    """
    token_ids = token_ids[:max_length]

    padding_needed = max_length - len(token_ids)

    if padding_needed > 0:
        token_ids = token_ids + [pad_id] * padding_needed

    return token_ids


# Test padding on our sample
padded_sample = pad_sequence(
    sample_ids,
    SOURCE_MAX_LENGTH_BPE,
    BPE_PAD_ID
)

print("\nOriginal length:", len(sample_ids))
print("Padded length:", len(padded_sample))
print("Number of PAD tokens:", padded_sample.count(BPE_PAD_ID))

BPE special token IDs
------------------------------
PAD: 0
UNK: 1
BOS: 2
EOS: 3
SEP: 4

Original length: 217
Padded length: 512
Number of PAD tokens: 295


In [10]:
def prepare_bpe_target(text):
    """
    Convert target text into BPE token IDs.

    Format:
        <BOS> + BPE tokens + <EOS>

    The sequence is truncated to leave room
    for both special tokens.
    """
    encoding = bpe_tokenizer.encode(str(text))

    token_ids = encoding.ids[:TARGET_MAX_LENGTH_BPE - 2]

    return [BPE_BOS_ID] + token_ids + [BPE_EOS_ID]


# Test on one training example
sample_target_ids = prepare_bpe_target(
    train_df.iloc[0]["target"]
)

print("Target length:", len(sample_target_ids))
print("First 10 IDs:", sample_target_ids[:10])
print("Last 10 IDs:", sample_target_ids[-10:])

print("\nFirst token ID:", sample_target_ids[0])
print("Last token ID:", sample_target_ids[-1])

Target length: 78
First 10 IDs: [2, 5261, 1560, 2055, 131, 62, 1427, 412, 21, 1516]
Last 10 IDs: [1560, 2055, 924, 176, 1784, 131, 337, 2252, 16, 3]

First token ID: 2
Last token ID: 3


In [11]:
import torch


def encode_and_pad_source(text):
    token_ids = encode_bpe_text(
        text,
        SOURCE_MAX_LENGTH_BPE
    )

    return pad_sequence(
        token_ids,
        SOURCE_MAX_LENGTH_BPE,
        BPE_PAD_ID
    )


def encode_and_pad_target(text):
    token_ids = prepare_bpe_target(text)

    return pad_sequence(
        token_ids,
        TARGET_MAX_LENGTH_BPE,
        BPE_PAD_ID
    )


def prepare_bpe_dataset(dataframe):
    sources = []
    decoder_inputs = []
    decoder_targets = []

    for text in dataframe["source"]:
        source_ids = encode_and_pad_source(text)
        sources.append(source_ids)

    for text in dataframe["target"]:
        target_ids = encode_and_pad_target(text)

        decoder_input = target_ids[:-1]
        decoder_target = target_ids[1:]

        decoder_inputs.append(decoder_input)
        decoder_targets.append(decoder_target)

    return {
        "source": torch.tensor(sources, dtype=torch.long),
        "decoder_input": torch.tensor(decoder_inputs, dtype=torch.long),
        "decoder_target": torch.tensor(decoder_targets, dtype=torch.long)
    }


# Create BPE datasets
bpe_train_data = prepare_bpe_dataset(train_df)
bpe_val_data = prepare_bpe_dataset(val_df)
bpe_test_data = prepare_bpe_dataset(test_df)


print("BPE dataset shapes")
print("-" * 40)

print("Train source:", bpe_train_data["source"].shape)
print("Train decoder input:", bpe_train_data["decoder_input"].shape)
print("Train decoder target:", bpe_train_data["decoder_target"].shape)

print("\nValidation source:", bpe_val_data["source"].shape)
print("Validation decoder input:", bpe_val_data["decoder_input"].shape)
print("Validation decoder target:", bpe_val_data["decoder_target"].shape)

print("\nTest source:", bpe_test_data["source"].shape)
print("Test decoder input:", bpe_test_data["decoder_input"].shape)
print("Test decoder target:", bpe_test_data["decoder_target"].shape)

BPE dataset shapes
----------------------------------------
Train source: torch.Size([800, 512])
Train decoder input: torch.Size([800, 127])
Train decoder target: torch.Size([800, 127])

Validation source: torch.Size([100, 512])
Validation decoder input: torch.Size([100, 127])
Validation decoder target: torch.Size([100, 127])

Test source: torch.Size([100, 512])
Test decoder input: torch.Size([100, 127])
Test decoder target: torch.Size([100, 127])


In [12]:
# Save BPE tokenizer
bpe_tokenizer_path = processed_dir / "bpe_tokenizer.json"
bpe_tokenizer.save(str(bpe_tokenizer_path))


# Save BPE datasets
bpe_train_path = processed_dir / "bpe_train_tensors.pt"
bpe_val_path = processed_dir / "bpe_val_tensors.pt"
bpe_test_path = processed_dir / "bpe_test_tensors.pt"

torch.save(bpe_train_data, bpe_train_path)
torch.save(bpe_val_data, bpe_val_path)
torch.save(bpe_test_data, bpe_test_path)


print("Saved BPE files")
print("-" * 40)

print("Tokenizer:", bpe_tokenizer_path.exists())
print("Train tensors:", bpe_train_path.exists())
print("Validation tensors:", bpe_val_path.exists())
print("Test tensors:", bpe_test_path.exists())

Saved BPE files
----------------------------------------
Tokenizer: True
Train tensors: True
Validation tensors: True
Test tensors: True


In [13]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 16

train_dataset_bpe = TensorDataset(
    bpe_train_data["source"],
    bpe_train_data["decoder_input"],
    bpe_train_data["decoder_target"]
)

val_dataset_bpe = TensorDataset(
    bpe_val_data["source"],
    bpe_val_data["decoder_input"],
    bpe_val_data["decoder_target"]
)

test_dataset_bpe = TensorDataset(
    bpe_test_data["source"],
    bpe_test_data["decoder_input"],
    bpe_test_data["decoder_target"]
)


train_loader_bpe = DataLoader(
    train_dataset_bpe,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader_bpe = DataLoader(
    val_dataset_bpe,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_bpe = DataLoader(
    test_dataset_bpe,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("BPE DataLoaders")
print("-" * 40)
print("Training batches:", len(train_loader_bpe))
print("Validation batches:", len(val_loader_bpe))
print("Test batches:", len(test_loader_bpe))

BPE DataLoaders
----------------------------------------
Training batches: 50
Validation batches: 7
Test batches: 7


In [14]:
# BPE Transformer configuration

VOCAB_SIZE_BPE = bpe_tokenizer.get_vocab_size()

D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
D_FF = 512
DROPOUT = 0.1

SOURCE_MAX_LENGTH = SOURCE_MAX_LENGTH_BPE
TARGET_MAX_LENGTH = TARGET_MAX_LENGTH_BPE

print("BPE Transformer Configuration")
print("-" * 45)
print("Vocabulary size:", VOCAB_SIZE_BPE)
print("Model dimension:", D_MODEL)
print("Attention heads:", NUM_HEADS)
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Decoder layers:", NUM_DECODER_LAYERS)
print("Feed-forward dimension:", D_FF)
print("Dropout:", DROPOUT)
print("Source max length:", SOURCE_MAX_LENGTH)
print("Target max length:", TARGET_MAX_LENGTH)
print("Head dimension:", D_MODEL // NUM_HEADS)

BPE Transformer Configuration
---------------------------------------------
Vocabulary size: 8000
Model dimension: 128
Attention heads: 4
Encoder layers: 2
Decoder layers: 2
Feed-forward dimension: 512
Dropout: 0.1
Source max length: 512
Target max length: 128
Head dimension: 32


In [15]:
print("Checking Transformer classes...")

classes_to_check = [
    "PositionalEncoding",
    "MultiHeadAttention",
    "FeedForwardNetwork",
    "TransformerEncoderBlock",
    "TransformerEncoder",
    "TransformerDecoderBlock",
    "TransformerDecoder",
    "BioTranscribeTransformer"
]

for class_name in classes_to_check:
    print(f"{class_name}: {class_name in globals()}")

Checking Transformer classes...
PositionalEncoding: False
MultiHeadAttention: False
FeedForwardNetwork: False
TransformerEncoderBlock: False
TransformerEncoder: False
TransformerDecoderBlock: False
TransformerDecoder: False
BioTranscribeTransformer: False


In [16]:
import nbformat
import torch
import torch.nn as nn
import math

notebook_path = "../notebooks/04_transformer_from_scratch.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = nbformat.read(f, as_version=4)

loaded_classes = []

for cell in notebook.cells:
    if cell.cell_type == "code" and "class " in cell.source:
        exec(cell.source, globals())

        for line in cell.source.splitlines():
            if line.strip().startswith("class "):
                class_name = line.strip().split("class ")[1].split("(")[0].split(":")[0].strip()
                loaded_classes.append(class_name)

print("Loaded Transformer classes:")
for name in loaded_classes:
    print("✓", name)

print("\nBioTranscribeTransformer available:",
      "BioTranscribeTransformer" in globals())

Loaded Transformer classes:
✓ PositionalEncoding
✓ MultiHeadAttention
✓ MultiHeadAttention
✓ FeedForwardNetwork
✓ TransformerEncoderBlock
✓ TransformerEncoder
✓ TransformerDecoderBlock
✓ TransformerDecoder
✓ BioTranscribeTransformer

BioTranscribeTransformer available: True


In [17]:
import nbformat

with open("04_transformer_from_scratch.ipynb", "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

nb = nbformat.validator.normalize(nb)[1]

with open("04_transformer_from_scratch.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Notebook normalized successfully.")

Notebook normalized successfully.


In [18]:
# Create the BPE Transformer model

bpe_model = BioTranscribeTransformer(
    vocab_size=VOCAB_SIZE_BPE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    source_max_length=SOURCE_MAX_LENGTH_BPE,
    target_max_length=TARGET_MAX_LENGTH_BPE,
    pad_id=BPE_PAD_ID,
    dropout=DROPOUT
)

print("BPE Transformer created successfully.")

total_params = sum(
    p.numel() for p in bpe_model.parameters() if p.requires_grad
)

print("Total trainable parameters:", total_params)

BPE Transformer created successfully.
Total trainable parameters: 4005696


In [19]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 16

bpe_train_dataset = TensorDataset(
    bpe_train_data["source"],
    bpe_train_data["decoder_input"],
    bpe_train_data["decoder_target"]
)

bpe_val_dataset = TensorDataset(
    bpe_val_data["source"],
    bpe_val_data["decoder_input"],
    bpe_val_data["decoder_target"]
)

bpe_test_dataset = TensorDataset(
    bpe_test_data["source"],
    bpe_test_data["decoder_input"],
    bpe_test_data["decoder_target"]
)

bpe_train_loader = DataLoader(
    bpe_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

bpe_val_loader = DataLoader(
    bpe_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

bpe_test_loader = DataLoader(
    bpe_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("BPE DataLoaders recreated successfully.")
print("Train batches:", len(bpe_train_loader))
print("Validation batches:", len(bpe_val_loader))
print("Test batches:", len(bpe_test_loader))

BPE DataLoaders recreated successfully.
Train batches: 50
Validation batches: 7
Test batches: 7


In [20]:
# Verify BPE Transformer forward pass

bpe_model.eval()

source_batch, decoder_input_batch, decoder_target_batch = next(
    iter(bpe_train_loader)
)

with torch.no_grad():
    output = bpe_model(
        source_batch,
        decoder_input_batch
    )

logits = output["logits"]

print("Source shape:", source_batch.shape)
print("Decoder input shape:", decoder_input_batch.shape)
print("Decoder target shape:", decoder_target_batch.shape)
print("Logits shape:", logits.shape)

print("\nExpected logits shape:",
      (source_batch.size(0),
       decoder_input_batch.size(1),
       VOCAB_SIZE_BPE))

print("\nOutput keys:", output.keys())

Source shape: torch.Size([16, 512])
Decoder input shape: torch.Size([16, 127])
Decoder target shape: torch.Size([16, 127])
Logits shape: torch.Size([16, 127, 8000])

Expected logits shape: (16, 127, 8000)

Output keys: dict_keys(['logits', 'encoder_attention', 'decoder_self_attention', 'decoder_cross_attention'])


In [21]:
# Loss function and optimizer for BPE Transformer

criterion_bpe = nn.CrossEntropyLoss(ignore_index=BPE_PAD_ID)

optimizer_bpe = torch.optim.Adam(
    bpe_model.parameters(),
    lr=1e-4
)

print("Loss function:", criterion_bpe)
print("Optimizer:", optimizer_bpe)
print("Learning rate:", optimizer_bpe.param_groups[0]["lr"])

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.0001


In [22]:
# Training and validation functions for BPE Transformer

def train_one_epoch_bpe(model, loader, criterion, optimizer, device="cpu"):
    model.train()
    total_loss = 0.0

    for source, decoder_input, decoder_target in loader:
        source = source.to(device)
        decoder_input = decoder_input.to(device)
        decoder_target = decoder_target.to(device)

        optimizer.zero_grad()

        output = model(source, decoder_input)
        logits = output["logits"]

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            decoder_target.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate_bpe(model, loader, criterion, device="cpu"):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for source, decoder_input, decoder_target in loader:
            source = source.to(device)
            decoder_input = decoder_input.to(device)
            decoder_target = decoder_target.to(device)

            output = model(source, decoder_input)
            logits = output["logits"]

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                decoder_target.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(loader)


# Quick 1-epoch test
train_loss = train_one_epoch_bpe(
    bpe_model,
    bpe_train_loader,
    criterion_bpe,
    optimizer_bpe
)

val_loss = validate_bpe(
    bpe_model,
    bpe_val_loader,
    criterion_bpe
)

print(f"Train Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

Train Loss: 8.6115
Validation Loss: 8.2732


In [ ]:
# Train BPE Transformer with early stopping

def train_bpe_with_early_stopping(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    max_epochs=30,
    patience=3,
    min_delta=0.001
):
    train_history = []
    val_history = []

    best_val_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(max_epochs):

        train_loss = train_one_epoch_bpe(
            model,
            train_loader,
            criterion,
            optimizer
        )

        val_loss = validate_bpe(
            model,
            val_loader,
            criterion
        )

        train_history.append(train_loss)
        val_history.append(val_loss)

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_model_state = {
                key: value.detach().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch + 1:02d}/{max_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

        if epochs_without_improvement >= patience:
            print("\nEarly stopping triggered.")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    print(f"\nBest Validation Loss: {best_val_loss:.4f}")

    return train_history, val_history


bpe_train_history, bpe_val_history = train_bpe_with_early_stopping(
    bpe_model,
    bpe_train_loader,
    bpe_val_loader,
    criterion_bpe,
    optimizer_bpe,
    max_epochs=30,
    patience=3
)

Epoch 01/30 | Train Loss: 8.0254 | Val Loss: 7.8453
Epoch 02/30 | Train Loss: 7.5922 | Val Loss: 7.4998
Epoch 03/30 | Train Loss: 7.2581 | Val Loss: 7.2593
Epoch 04/30 | Train Loss: 7.0372 | Val Loss: 7.1224
Epoch 05/30 | Train Loss: 6.9060 | Val Loss: 7.0551
Epoch 06/30 | Train Loss: 6.8292 | Val Loss: 7.0124
Epoch 07/30 | Train Loss: 6.7665 | Val Loss: 6.9785
Epoch 08/30 | Train Loss: 6.7123 | Val Loss: 6.9495
Epoch 09/30 | Train Loss: 6.6634 | Val Loss: 6.9222
Epoch 10/30 | Train Loss: 6.6137 | Val Loss: 6.8969


In [ ]:
# Continue BPE training for 10 more epochs

for epoch in range(10):

    train_loss = train_one_epoch_bpe(
        bpe_model,
        bpe_train_loader,
        criterion_bpe,
        optimizer_bpe
    )

    val_loss = validate_bpe(
        bpe_model,
        bpe_val_loader,
        criterion_bpe
    )

    bpe_train_history.append(train_loss)
    bpe_val_history.append(val_loss)

    print(
        f"Epoch {len(bpe_train_history):02d}/40 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 31/40 | Train Loss: 5.6084 | Val Loss: 6.5562
Epoch 32/40 | Train Loss: 5.5694 | Val Loss: 6.5535
Epoch 33/40 | Train Loss: 5.5205 | Val Loss: 6.5450
Epoch 34/40 | Train Loss: 5.4794 | Val Loss: 6.5355
Epoch 35/40 | Train Loss: 5.4335 | Val Loss: 6.5308
Epoch 36/40 | Train Loss: 5.3894 | Val Loss: 6.5315
Epoch 37/40 | Train Loss: 5.3442 | Val Loss: 6.5242
Epoch 38/40 | Train Loss: 5.3032 | Val Loss: 6.5236
Epoch 39/40 | Train Loss: 5.2573 | Val Loss: 6.5193
Epoch 40/40 | Train Loss: 5.2156 | Val Loss: 6.5222


In [ ]:
# Evaluate BPE model on the test set

bpe_test_loss = validate_bpe(
    bpe_model,
    bpe_test_loader,
    criterion_bpe
)

print(f"BPE Test Loss: {bpe_test_loss:.4f}")

BPE Test Loss: 6.4686


In [ ]:
def generate_bpe_greedy(model, source, max_length=127):
    model.eval()

    source = source.unsqueeze(0)

    decoder_input = torch.tensor(
        [[BPE_BOS_ID]],
        dtype=torch.long
    )

    generated_tokens = []

    with torch.no_grad():
        for _ in range(max_length):
            output = model(source, decoder_input)
            logits = output["logits"]

            next_token = logits[:, -1, :].argmax(dim=-1)

            decoder_input = torch.cat(
                [decoder_input, next_token.unsqueeze(1)],
                dim=1
            )

            token_id = next_token.item()

            if token_id == BPE_EOS_ID:
                break

            generated_tokens.append(token_id)

    return generated_tokens


# Generate an answer for the first test example
source_sample = bpe_test_data["source"][0]

generated_ids = generate_bpe_greedy(
    bpe_model,
    source_sample
)

generated_text = bpe_tokenizer.decode(generated_ids)

print("Generated BPE answer:")
print(generated_text)

print("\nActual answer:")
print(test_df.iloc[0]["target"])

Generated BPE answer:
The results of the study , the first - based on the development of the first - up , but a significant difference in the development of the first - term of the use of the first - term of the first - up of the use of the first - term of the first - up of the first - based on the use of the use of the use of the use of the use of the use of the use of the use of the first - term .

Actual answer:
Having a regular primary care clinician is embraced in pediatrics, although team care among physicians is also widely practiced. For young children, having a regular clinician is associated with modest gains in interpersonal quality and no differences in content of anticipatory guidance. The benefit of having a regular clinician may primarily occur in interpersonal quality for subgroups of young children.


In [ ]:
def generate_bpe_sampling(
    model,
    source,
    max_length=127,
    temperature=0.8,
    top_k=20
):
    model.eval()

    source = source.unsqueeze(0)

    decoder_input = torch.tensor(
        [[BPE_BOS_ID]],
        dtype=torch.long
    )

    generated_tokens = []

    with torch.no_grad():
        for _ in range(max_length):

            output = model(source, decoder_input)
            logits = output["logits"][:, -1, :]

            # Temperature
            logits = logits / temperature

            # Top-k filtering
            if top_k is not None:
                values, indices = torch.topk(logits, top_k)
                filtered_logits = torch.full_like(
                    logits,
                    float("-inf")
                )
                filtered_logits.scatter_(1, indices, values)
                logits = filtered_logits

            probabilities = torch.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            token_id = next_token.item()

            if token_id == BPE_EOS_ID:
                break

            generated_tokens.append(token_id)

            decoder_input = torch.cat(
                [decoder_input, next_token],
                dim=1
            )

    return generated_tokens


source_sample = bpe_test_data["source"][0]

generated_ids = generate_bpe_sampling(
    bpe_model,
    source_sample,
    temperature=0.8,
    top_k=20
)

generated_text = bpe_tokenizer.decode(generated_ids)

print("BPE Sampling Answer:")
print(generated_text)

print("\nActual Answer:")
print(test_df.iloc[0]["target"])

BPE Sampling Answer:
There and can be a higher in patients with a limited , this study , but a new - term are needed to the development of the clinical use of H and patient .

Actual Answer:
Having a regular primary care clinician is embraced in pediatrics, although team care among physicians is also widely practiced. For young children, having a regular clinician is associated with modest gains in interpersonal quality and no differences in content of anticipatory guidance. The benefit of having a regular clinician may primarily occur in interpersonal quality for subgroups of young children.


In [ ]:
def add_sep_to_source(text):
    """
    Insert <SEP> between the question and context.
    """
    text = str(text)

    if " context: " in text:
        question_part, context_part = text.split(" context: ", 1)
        return question_part + " <SEP> context: " + context_part

    return text


train_df_sep = train_df.copy()
val_df_sep = val_df.copy()
test_df_sep = test_df.copy()

train_df_sep["source"] = train_df_sep["source"].apply(add_sep_to_source)
val_df_sep["source"] = val_df_sep["source"].apply(add_sep_to_source)
test_df_sep["source"] = test_df_sep["source"].apply(add_sep_to_source)

print(train_df_sep["source"].iloc[0][:500])

question: Visceral adipose tissue area measurement at a single level: can it represent visceral adipose tissue volume? <SEP> context: Measurement of visceral adipose tissue (VAT) needs to be accurate and sensitive to change for risk monitoring. The purpose of this study is to determine the CT slice location where VAT area can best reflect changes in VAT volume and body weight. 60 plain abdominal CT images from 30 males [mean age (range) 51 (41-68) years, mean body weight (range) 71.1 (101.9-50.9


In [ ]:
def prepare_bpe_sep_dataset(dataframe):
    sources = []
    decoder_inputs = []
    decoder_targets = []

    for text in dataframe["source"]:
        token_ids = encode_bpe_text(
            text,
            SOURCE_MAX_LENGTH_BPE
        )
        sources.append(
            pad_sequence(
                token_ids,
                SOURCE_MAX_LENGTH_BPE,
                BPE_PAD_ID
            )
        )

    for text in dataframe["target"]:
        target_ids = encode_and_pad_target(text)

        decoder_inputs.append(target_ids[:-1])
        decoder_targets.append(target_ids[1:])

    return {
        "source": torch.tensor(sources, dtype=torch.long),
        "decoder_input": torch.tensor(decoder_inputs, dtype=torch.long),
        "decoder_target": torch.tensor(decoder_targets, dtype=torch.long)
    }


bpe_sep_train_data = prepare_bpe_sep_dataset(train_df_sep)
bpe_sep_val_data = prepare_bpe_sep_dataset(val_df_sep)
bpe_sep_test_data = prepare_bpe_sep_dataset(test_df_sep)

print("Train:", bpe_sep_train_data["source"].shape,
      bpe_sep_train_data["decoder_input"].shape,
      bpe_sep_train_data["decoder_target"].shape)

print("Validation:", bpe_sep_val_data["source"].shape,
      bpe_sep_val_data["decoder_input"].shape,
      bpe_sep_val_data["decoder_target"].shape)

print("Test:", bpe_sep_test_data["source"].shape,
      bpe_sep_test_data["decoder_input"].shape,
      bpe_sep_test_data["decoder_target"].shape)

Train: torch.Size([800, 512]) torch.Size([800, 127]) torch.Size([800, 127])
Validation: torch.Size([100, 512]) torch.Size([100, 127]) torch.Size([100, 127])
Test: torch.Size([100, 512]) torch.Size([100, 127]) torch.Size([100, 127])


In [ ]:
BATCH_SIZE = 16

bpe_sep_train_dataset = TensorDataset(
    bpe_sep_train_data["source"],
    bpe_sep_train_data["decoder_input"],
    bpe_sep_train_data["decoder_target"]
)

bpe_sep_val_dataset = TensorDataset(
    bpe_sep_val_data["source"],
    bpe_sep_val_data["decoder_input"],
    bpe_sep_val_data["decoder_target"]
)

bpe_sep_test_dataset = TensorDataset(
    bpe_sep_test_data["source"],
    bpe_sep_test_data["decoder_input"],
    bpe_sep_test_data["decoder_target"]
)

bpe_sep_train_loader = DataLoader(
    bpe_sep_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

bpe_sep_val_loader = DataLoader(
    bpe_sep_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

bpe_sep_test_loader = DataLoader(
    bpe_sep_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(bpe_sep_train_loader))
print("Validation batches:", len(bpe_sep_val_loader))
print("Test batches:", len(bpe_sep_test_loader))

Train batches: 50
Validation batches: 7
Test batches: 7


In [ ]:
bpe_sep_model = BioTranscribeTransformer(
    vocab_size=VOCAB_SIZE_BPE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    source_max_length=SOURCE_MAX_LENGTH_BPE,
    target_max_length=TARGET_MAX_LENGTH_BPE,
    pad_id=BPE_PAD_ID,
    dropout=DROPOUT
)

total_params = sum(
    p.numel()
    for p in bpe_sep_model.parameters()
    if p.requires_grad
)

print("BPE + <SEP> Transformer created successfully.")
print(f"Total trainable parameters: {total_params}")

BPE + <SEP> Transformer created successfully.
Total trainable parameters: 4005696


In [ ]:
import torch
import torch.nn as nn

criterion_bpe_sep = nn.CrossEntropyLoss(
    ignore_index=BPE_PAD_ID
)

optimizer_bpe_sep = torch.optim.Adam(
    bpe_sep_model.parameters(),
    lr=1e-4
)

print("Loss function and optimizer created successfully.")

NameError: name 'BPE_PAD_ID' is not defined

In [ ]:
#Set loss and optimizer
criterion_bpe_sep = nn.CrossEntropyLoss(
    ignore_index=BPE_PAD_ID
)

optimizer_bpe_sep = torch.optim.Adam(
    bpe_sep_model.parameters(),
    lr=1e-4
)

print("Loss function:", criterion_bpe_sep)
print("Optimizer:", optimizer_bpe_sep)

NameError: name 'nn' is not defined